In [1]:
import datacube
import numpy as np
from shapely.geometry import box

dc = datacube.Datacube(app="ndvi_cloud_check")

tile = "p115r078"
product = "ga_ls8c_ard_3"   # try ls9 too
date = "20230905"          # change to test
lon_min, lon_max = 112.31, 114.601
lat_min, lat_max = -26.932, -25.049

t0 = f"{date[:4]}-{date[4:6]}-{date[6:]}"
time = (t0, t0)

datasets = dc.find_datasets(
    product=product,
    lon=(lon_min, lon_max),
    lat=(lat_min, lat_max),
    time=time,
)

print("datasets found:", len(datasets))

def get_cloud_meta(ds):
    md = getattr(ds, "metadata_doc", {}) or {}
    cloud = None
    for key in ("eo:cloud_cover", "cloud_cover", "landsat:cloud_cover"):
        if key in md:
            try:
                cloud = float(md[key]); return cloud
            except: pass
    props = md.get("properties", {}) or {}
    if "eo:cloud_cover" in props:
        try:
            cloud = float(props["eo:cloud_cover"]); return cloud
        except: pass
    return None

for i, ds in enumerate(datasets[:10]):
    cloud = get_cloud_meta(ds)
    uris = getattr(ds, "uris", []) or []
    print(f"[{i}] center_time={ds.center_time} cloud_meta={cloud} uri0={uris[0] if uris else None}")

# Now load oa_fmask for the bbox and compute clear%
ds = dc.load(
    product=product,
    measurements=["oa_fmask"],
    time=time,
    lon=(lon_min, lon_max),
    lat=(lat_min, lat_max),
    dask_chunks={"x": 2048, "y": 2048},
    skip_broken_datasets=True,
)

oa = ds["oa_fmask"].isel(time=0)
nodata = oa.attrs.get("nodata", None)
print("oa_fmask nodata attr:", nodata)

CLEAR_VALUES = (0,)  # your current pipeline assumption

if nodata is None:
    valid = np.isfinite(oa)
else:
    valid = (oa != nodata)

clear = oa.isin(CLEAR_VALUES) & valid
clear_pct = float((clear.sum() / valid.sum()).compute().values * 100.0)

vals, counts = np.unique(oa.compute().values, return_counts=True)
print("unique oa_fmask values (sampled bbox):")
for v, c in zip(vals.tolist(), counts.tolist()):
    print(f"  {v}: {c}")

print(f"clear% (oa_fmask in {CLEAR_VALUES}, valid-only): {clear_pct:.2f}%")


datasets found: 3
[0] center_time=2023-09-05 02:21:58.869079+00:00 cloud_meta=1.245074850916059 uri0=s3://dea-public-data/baseline/ga_ls8c_ard_3/115/077/2023/09/05/ga_ls8c_ard_3-2-1_115077_2023-09-05_final.odc-metadata.yaml
[1] center_time=2023-09-05 02:22:22.823603+00:00 cloud_meta=14.502057579348227 uri0=s3://dea-public-data/baseline/ga_ls8c_ard_3/115/078/2023/09/05/ga_ls8c_ard_3-2-1_115078_2023-09-05_final.odc-metadata.yaml
[2] center_time=2023-09-05 02:22:46.767509+00:00 cloud_meta=25.58541761628773 uri0=s3://dea-public-data/baseline/ga_ls8c_ard_3/115/079/2023/09/05/ga_ls8c_ard_3-2-1_115079_2023-09-05_final.odc-metadata.yaml


/tmp/ipykernel_19722/594843877.py:42: ODC2DeprecationWarning: Call to deprecated function (or staticmethod) uris. (Multiple locations are now deprecated. Please use the 'uri' attribute instead.)
-- Deprecated since version 1.9.0.
  uris = getattr(ds, "uris", []) or []


oa_fmask nodata attr: 0


/env/lib/python3.12/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


unique oa_fmask values (sampled bbox):
  0: 57159542
  1: 5449514
  2: 216334
  3: 165873
  5: 5444895
clear% (oa_fmask in (0,), valid-only): 0.00%
